## Phase 0 — Already Done

- data collection
  - NERRS (1995–2025, most regions minus hudson river and 2x great lakes, all stations)
    - originally 15min resolution
    - mostly intact water conditions
    - very sparse nutrient data
      - nutrient data extended ("as-of" forward-fill, 7-day cap)
      - this just in case a record at 945am was skipped over by picking the 1000am record instead to collate at 1hr resolution
    - entirely blank meteorlogical data?
  - ERA5 atmospherics
    - get that atmospheric data back
    - 1hr resolution
    - 0.25 degree resolution
      - not a perfect match to each station in nerrs
      - median centroid was calculated for all stations in a region
      - though most times that centroid ended up nearby but over land (so switched to skin temp instead of water surface temp)
- data cleaning and collation
- split into two datasets
  - full (with nutrients, less complete timeline)
  - and core (without nutrients, more complete)
  - script can be rerun for 1hr resolution
  - but 12hr resolution is included in this repo (zipped)
- various metrics analysis, exploration

In [ ]:
# first... dependencies
import numpy as np                  # for arrays and math
import pandas as pd                 # for dataframes and csv I/O
import matplotlib.pyplot as plt     # basic visualizations
import seaborn as sns               # for quick readable charts
from sklearn.cluster import KMeans      # simple clustering
from sklearn.decomposition import PCA   # quick 2d display of clusters
from sklearn.preprocessing import StandardScaler  # normalize features
from sklearn.metrics import silhouette_score  # cluster quality check

# keep charts easy to read
sns.set_theme( style = 'whitegrid' )

# the files (thee have been collated and cleaned already)
res = 1 # hours - alternatives 4 and 12

# https://www.youtube.com/watch?v=_W7K79FjI58
# mandatory listening while working on this project



In [ ]:
water = pd.read_csv( f'../data/{res}hr/t4d.{res}hr.water.history.csv' )

water[ 'region' ] = water[ 'region' ].astype( str ).str.strip( ).str.lower( )
water[ 'station' ] = water[ 'station' ].astype( str ).str.strip( ).str.lower( )

# HEE is too sparse for this analysis, so drop it globally here
water = water.loc[ water[ 'region' ] != 'hee' ].copy( )


In [ ]:
# lightweight station and region lookup for labels only
station_lookup = pd.read_csv( '../data/reference/nerrs_station_index.csv' )

station_lookup[ 'region' ] = station_lookup[ 'region_code' ].astype( str ).str.strip( ).str.lower( )
station_lookup[ 'station_full' ] = station_lookup[ 'station' ].astype( str ).str.strip( ).str.lower( )
station_lookup[ 'station' ] = station_lookup[ 'station_full' ].str[ -2: ]

station_lookup = station_lookup[ [ 
    'region',
    'station',
    'station_full',
    'region_name',
    'station_name',
    'latitude',
    'longitude',
    'in_t4d_1hr_water_history',
] ].drop_duplicates( subset = [ 'region', 'station' ] )

region_name_lookup = ( 
    station_lookup
    .dropna( subset = [ 'region_name' ] )
    .drop_duplicates( subset = [ 'region' ] )
    .set_index( 'region' )[ 'region_name' ]
    .to_dict( )
)

station_name_lookup = ( 
    station_lookup
    .dropna( subset = [ 'station_name' ] )
    .set_index( [ 'region', 'station' ] )[ 'station_name' ]
    .to_dict( )
)


def station_label( region, station ):
    region_key = str( region ).strip( ).lower( )
    station_key = str( station ).strip( ).lower( )[ -2: ]

    name = station_name_lookup.get( ( region_key, station_key ) )

    if pd.isna( name ) or name is None or str( name ).strip( ) == '':
        return station_key

    return f'{station_key} - {name}'

lookup_missing = ( 
    water[ [ 'region', 'station' ] ]
    .drop_duplicates( )
    .merge( station_lookup[ [ 'region', 'station', 'station_name' ] ], on = [ 'region', 'station' ], how = 'left' )
    [ 'station_name' ]
    .isna( )
    .sum( )
)

print( f'lookup missing station names: {int( lookup_missing )}' )



## Phase 1 — Characterization & Classification
Goal: understand what we have before modeling anything

- 1.1 compute per-station summary statistics over a defined baseline period (suggest 1995–2005)
  - mean annual water temp, seasonal amplitude, mean salinity, mean DO
- 1.2 cluster stations in (mean temp × seasonal amplitude) space 
  - k-means, try k=3 or 4, use elbow/silhouette to let the data suggest the right number of groups
  - example in nutrient analysis work
- 1.3 enrich clusters with any available station metadata (estuary type, distance from mouth, watershed area)
  - confirm clusters are physically meaningful, not just statistical artifacts
- 1.4 assign each station a baseline regime label
  - see if kmeans self-identify... 
  - otherwise probably get a rolling means (temp) per station for 1995-2000 to classify
- 1.5 visualize stations on a map colored by regime
  - sanity check that PR/HI are warm, alaska/maine are cold, etc.
- 1.6 document baseline period statistics per regime as a reference table
  - this will be needed for the paper/poster later, too

In [ ]:
# 1.0 - a description
water.describe( ).round( 3 ).T

In [ ]:
# lets make this more readable
water = water.rename( columns = { 
    'w_temp_c': 'water_temp',
    'w_sal_psu': 'salinity',
    'w_do_mg_l': 'oxygen',
    'w_do_pct': 'oxy_saturation',
    'depth_m': 'depth',
    'w_ph': 'ph',
    'm_wind_ms': 'wind_speed',
    'm_ssrd_kwh_m2': 'solar_radiation',
    'm_precip_mmh': 'precipitation',
    'm_temo_c': 'air_temp'
} )

In [ ]:
# 1.1 - station character baseline (first valid years, not fixed 1995-2000)
# this keeps the idea simple: each station gets its own early baseline window

base_all = water.copy( )
base_all[ 'datetime' ] = pd.to_datetime( base_all[ 'datetime' ], errors = 'coerce' )

base_all = base_all.loc[ 
    :,
    [ 
        'region',
        'station',
        'datetime',
        'water_temp',
        'salinity',
        'oxygen',
        'oxy_saturation',
        'ph',
        'depth',
    ],
].dropna( subset = [ 'datetime' ] )

base_all[ 'year' ] = base_all[ 'datetime' ].dt.year

# annual coverage check so thin years do not define the baseline
year_obs = ( 
    base_all
    .groupby( [ 'region', 'station', 'year' ], as_index = False )
    .agg( n_obs = ( 'datetime', 'size' ) )
)

year_obs[ 'expected_obs' ] = np.where( 
    year_obs[ 'year' ] % 4 == 0,
    366 * 24,
    365 * 24,
)

year_obs[ 'coverage_frac' ] = year_obs[ 'n_obs' ] / year_obs[ 'expected_obs' ]
year_obs[ 'year_is_valid' ] = year_obs[ 'coverage_frac' ] >= 0.70

valid_years = year_obs.loc[ year_obs[ 'year_is_valid' ] ].copy( )
valid_years = valid_years.sort_values( [ 'region', 'station', 'year' ] ).reset_index( drop = True )
valid_years[ 'valid_year_rank' ] = valid_years.groupby( [ 'region', 'station' ] ).cumcount( ) + 1

# first 5 valid years per station
baseline_years = valid_years.loc[ valid_years[ 'valid_year_rank' ] <= 5 ].copy( )

baseline_window = ( 
    baseline_years
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        baseline_start_year = ( 'year', 'min' ),
        baseline_end_year = ( 'year', 'max' ),
        n_valid_years = ( 'year', 'size' ),
        mean_year_coverage = ( 'coverage_frac', 'mean' ),
    )
)

baseline_window[ 'is_partial_baseline' ] = baseline_window[ 'n_valid_years' ] < 5

# simple rule: keep stations with at least 3 valid years
eligible_stations = baseline_window.loc[ 
    baseline_window[ 'n_valid_years' ] >= 3,
    [ 'region', 'station' ],
]

# pull only rows from each station's selected baseline years
base = base_all.merge( 
    baseline_years[ [ 'region', 'station', 'year' ] ],
    on = [ 'region', 'station', 'year' ],
    how = 'inner',
)

# then keep only stations that passed the >=3-year minimum
base = base.merge( eligible_stations, on = [ 'region', 'station' ], how = 'inner' )


In [ ]:
# okay, now build station character features from the selected baseline window
base[ 'month' ] = base[ 'datetime' ].dt.month
# get mean of each year per station, then the mean of those five years per station
# yearly means first ( equal-year weighting )
annual = ( 
    base
    .groupby( [ 'region', 'station', 'year' ], as_index = False )
    .agg( 
        water_temp_ymean = ( 'water_temp', 'mean' ),
        salinity_ymean = ( 'salinity', 'mean' ),
        oxygen_ymean = ( 'oxygen', 'mean' ),
        saturation_ymean = ( 'oxy_saturation', 'mean' ),
        ph_ymean = ( 'ph', 'mean' ),
        depth_ymean = ( 'depth', 'mean' ),
    )
)

annual_means = ( 
    annual
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        mean_annual_water_temp = ( 'water_temp_ymean', 'mean' ),
        mean_annual_salinity = ( 'salinity_ymean', 'mean' ),
        mean_annual_oxygen = ( 'oxygen_ymean', 'mean' ),
        mean_annual_saturation = ( 'saturation_ymean', 'mean' ),
        mean_annual_ph = ( 'ph_ymean', 'mean' ),
        mean_annual_depth = ( 'depth_ymean', 'mean' ),
        n_years_water_temp = ( 'water_temp_ymean', 'count' ),
        n_years_salinity = ( 'salinity_ymean', 'count' ),
        n_years_oxygen = ( 'oxygen_ymean', 'count' ),
        n_years_saturation = ( 'saturation_ymean', 'count' ),
        n_years_ph = ( 'ph_ymean', 'count' ),
        n_years_depth = ( 'depth_ymean', 'count' ),
    )
)

# seasonal amplitudes from monthly climatology
monthly = ( 
    base
    .groupby( [ 'region', 'station', 'month' ], as_index = False )
    .agg( 
        water_temp_mmean = ( 'water_temp', 'mean' ),
        salinity_mmean = ( 'salinity', 'mean' ),
        oxygen_mmean = ( 'oxygen', 'mean' ),
        saturation_mmean = ( 'oxy_saturation', 'mean' ),
        ph_mmean = ( 'ph', 'mean' ),
        depth_mmean = ( 'depth', 'mean' ),
    )
)

# building to a new feature, the 'swing' the mean swing of seasonal properties
seasonal_amp = ( 
    monthly
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        seasonal_amp_water_temp = ( 'water_temp_mmean', lambda s: s.max( ) - s.min( ) ),
        seasonal_amp_salinity = ( 'salinity_mmean', lambda s: s.max( ) - s.min( ) ),
        seasonal_amp_oxygen = ( 'oxygen_mmean', lambda s: s.max( ) - s.min( ) ),
        seasonal_amp_saturation = ( 'saturation_mmean', lambda s: s.max( ) - s.min( ) ),
        seasonal_amp_ph = ( 'ph_mmean', lambda s: s.max( ) - s.min( ) ),
        seasonal_amp_depth = ( 'depth_mmean', lambda s: s.max( ) - s.min( ) ),
    )
)

# depth summary ( useful if sensor environment differs by station )
# let's see if the recorders are variable depth per station, or predictable?
depth_stats = ( 
    base
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        median_depth = ( 'depth', 'median' ),
        p10_depth = ( 'depth', lambda s: s.quantile( 0.10 ) ),
        p90_depth = ( 'depth', lambda s: s.quantile( 0.90 ) ),
    )
)
depth_stats[ 'iqr_depth' ] = depth_stats[ 'p90_depth' ] - depth_stats[ 'p10_depth' ]

coverage = ( 
    base
    .groupby( [ 'region', 'station' ], as_index = False )
    .agg( 
        n_obs_total = ( 'datetime', 'size' ),
        n_years_present = ( 'year', 'nunique' ),
    )
)

station_baseline = ( 
    annual_means
    .merge( seasonal_amp, on = [ 'region', 'station' ], how = 'left' )
    .merge( depth_stats, on = [ 'region', 'station' ], how = 'left' )
    .merge( coverage, on = [ 'region', 'station' ], how = 'left' )
    .merge( 
        baseline_window[ [ 
            'region',
            'station',
            'baseline_start_year',
            'baseline_end_year',
            'n_valid_years',
            'mean_year_coverage',
            'is_partial_baseline',
        ] ],
        on = [ 'region', 'station' ],
        how = 'left',
    )
    .sort_values( [ 'region', 'station' ] )
    .reset_index( drop = True )
)

# placeholder flag in case we later add nearest-neighbor feature fallback
station_baseline[ 'character_imputed' ] = False
station_baseline_display = ( 
    station_baseline
    .merge( 
        station_lookup[ [ 'region', 'station', 'region_name', 'station_name', 'latitude', 'longitude' ] ],
        on = [ 'region', 'station' ],
        how = 'left',
    )
    .sort_values( [ 'region', 'station' ] )
    .reset_index( drop = True )
)




In [ ]:
# quick baseline summary
n_total_stations = baseline_window[ [ 'region', 'station' ] ].drop_duplicates( ).shape[ 0 ]
n_eligible_stations = eligible_stations.shape[ 0 ]
n_full_five_year = int( ( baseline_window[ 'n_valid_years' ] >= 5 ).sum( ) )

print( f'stations with >=3 valid baseline years: {n_eligible_stations} of {n_total_stations}' )
print( f'stations with full 5-year baseline: {n_full_five_year} of {n_total_stations}' )

baseline_preview = ( 
    baseline_window
    .sort_values( [ 'region', 'station' ] )
    .head( 20 )
)

baseline_preview


In [ ]:
eligible_stations_display = ( 
    eligible_stations
    .merge( 
        station_lookup[ [ 'region', 'station', 'region_name', 'station_name', 'latitude', 'longitude' ] ],
        on = [ 'region', 'station' ],
        how = 'left',
    )
    .sort_values( [ 'region', 'station' ] )
    .reset_index( drop = True )
)

eligible_stations_display


In [ ]:
# tall small-multiples: one subplot per region, one line per station

plot_df = water.copy( )

plot_df[ 'datetime' ] = pd.to_datetime( plot_df[ 'datetime' ], errors = 'coerce' )
plot_df = plot_df.dropna( subset = [ 'datetime' ] )

temp_col = 'water_temp' if 'water_temp' in plot_df.columns else 'w_temp_c'

plot_df[ 'year' ] = plot_df[ 'datetime' ].dt.year

annual_station = ( 
    plot_df
    .groupby( [ 'region', 'station', 'year' ], as_index = False )[ temp_col ]
    .mean( )
    .rename( columns = { temp_col: 'mean_annual_temp' } )
)

regions = sorted( annual_station[ 'region' ].dropna( ).unique( ) )
n_regions = len( regions )

fig, axes = plt.subplots( 
    n_regions,
    1,
    figsize = ( 16, max( 3 * n_regions, 10 ) ),
    sharex = True,
    constrained_layout = True,
)

if n_regions == 1:
    axes = [ axes ]

for ax, region in zip( axes, regions ):
    sub = annual_station.loc[ annual_station[ 'region' ] == region ].sort_values( [ 'station', 'year' ] )

    for station, g in sub.groupby( 'station' ):
        ax.plot( 
            g[ 'year' ],
            g[ 'mean_annual_temp' ],
            linewidth = 1.4,
            alpha = 0.85,
            label = station_label( region, station ),
        )

    region_title = region_name_lookup.get( region, region )
    ax.set_title( f'{region.upper( )} ( {region_title} ): Mean Annual Water Temp by Station' )
    ax.set_ylabel( 'Temp ( C )' )

    # keep legends readable
    n_stations = sub[ 'station' ].nunique( )

    if n_stations <= 20:
        legend_cols = 2 if n_stations <= 12 else 3
        ax.legend( 
            ncol = legend_cols,
            fontsize = 7,
            frameon = False,
            loc = 'upper left',
            bbox_to_anchor = ( 1.01, 1.0 ),
            borderaxespad = 0,
        )

    else:
        ax.text( 
            0.99,
            0.95,
            f'{n_stations} stations',
            transform = ax.transAxes,
            ha = 'right',
            va = 'top',
            fontsize = 8,
        )

axes[ -1 ].set_xlabel( 'Year' )
plt.show( )




### 1.2 - KMeans on Station Character

cluster stations using the character features we just built


In [ ]:
# local import fallback so this cell runs even if imports were not re-run
from sklearn.metrics import silhouette_score

# use a small, readable set of station-character features
feature_cols = [ 
    'mean_annual_water_temp',
    'mean_annual_salinity',
    'mean_annual_oxygen',
    'mean_annual_saturation',
    'mean_annual_ph',
    'mean_annual_depth',
    'seasonal_amp_water_temp',
    'seasonal_amp_salinity',
    'seasonal_amp_oxygen',
    'seasonal_amp_saturation',
    'seasonal_amp_ph',
    'seasonal_amp_depth',
    'iqr_depth',
]

kmeans_input = station_baseline[ [ 'region', 'station' ] + feature_cols ].copy( )

# simple missing-value fill: region median, then global median
for col in feature_cols:
    kmeans_input[ col ] = kmeans_input.groupby( 'region' )[ col ].transform( lambda s: s.fillna( s.median( ) ) )
    kmeans_input[ col ] = kmeans_input[ col ].fillna( kmeans_input[ col ].median( ) )

# scale so no one feature dominates by units
scaler = StandardScaler( )
X_scaled = scaler.fit_transform( kmeans_input[ feature_cols ] )

# report elbow and silhouette for a simple K-range
k_scan_rows = [ ]

for k in range( 2, 11 ):
    km_scan = KMeans( n_clusters = k, random_state = 42, n_init = 20 )
    labels_scan = km_scan.fit_predict( X_scaled )

    k_scan_rows.append( { 
        'k': k,
        'inertia': float( km_scan.inertia_ ),
        'silhouette': float( silhouette_score( X_scaled, labels_scan ) ),
    } )

k_scan = pd.DataFrame( k_scan_rows )
k_scan[ 'inertia_drop' ] = k_scan[ 'inertia' ].shift( 1 ) - k_scan[ 'inertia' ]
k_scan[ 'inertia_drop_pct' ] = k_scan[ 'inertia_drop' ] / k_scan[ 'inertia' ].shift( 1 )

print( 'k scan ( elbow + silhouette ):' )
print( k_scan.round( 4 ) )

plt.figure( figsize = ( 10, 4 ) )
plt.plot( k_scan[ 'k' ], k_scan[ 'inertia' ], marker = 'o' )
plt.title( 'Elbow Plot: K vs Inertia' )
plt.xlabel( 'K' )
plt.ylabel( 'Inertia' )
plt.tight_layout( )
plt.show( )

plt.figure( figsize = ( 10, 4 ) )
plt.plot( k_scan[ 'k' ], k_scan[ 'silhouette' ], marker = 'o' )
plt.title( 'K vs Silhouette Score' )
plt.xlabel( 'K' )
plt.ylabel( 'Silhouette' )
plt.tight_layout( )
plt.show( )

# keep this simple and explicit for now
k_clusters = 4
kmeans_model = KMeans( n_clusters = k_clusters, random_state = 42, n_init = 20 )
kmeans_input[ 'cluster' ] = kmeans_model.fit_predict( X_scaled )

# update station tables (drop old cluster if this cell gets re-run)
if 'cluster' in station_baseline.columns:
    station_baseline = station_baseline.drop( columns = [ 'cluster' ] )

station_baseline = station_baseline.merge( 
    kmeans_input[ [ 'region', 'station', 'cluster' ] ],
    on = [ 'region', 'station' ],
    how = 'left',
)

if 'cluster' in station_baseline_display.columns:
    station_baseline_display = station_baseline_display.drop( columns = [ 'cluster' ] )

station_baseline_display = station_baseline_display.merge( 
    kmeans_input[ [ 'region', 'station', 'cluster' ] ],
    on = [ 'region', 'station' ],
    how = 'left',
)

print( 'cluster sizes:' )
print( station_baseline[ 'cluster' ].value_counts( ).sort_index( ) )

cluster_profile = station_baseline.groupby( 'cluster' )[ feature_cols ].mean( ).round( 3 )

station_baseline_display[ [ 
    'region',
    'station',
    'station_name',
    'cluster',
    'baseline_start_year',
    'baseline_end_year',
    'n_valid_years',
] ].sort_values( [ 'cluster', 'region', 'station' ] ).head( 40 )

# quick 2d plot so we can see cluster separation
pca = PCA( n_components = 2 )
pcs = pca.fit_transform( X_scaled )

cluster_plot = kmeans_input[ [ 'region', 'station', 'cluster' ] ].copy( )
cluster_plot[ 'pc1' ] = pcs[ :, 0 ]
cluster_plot[ 'pc2' ] = pcs[ :, 1 ]

plt.figure( figsize = ( 11, 7 ) )
sns.scatterplot( 
    data = cluster_plot,
    x = 'pc1',
    y = 'pc2',
    hue = 'cluster',
    palette = 'tab10',
    s = 70,
    alpha = 0.9,
)
plt.title( 'KMeans Clusters of Station Character ( PCA View )' )
plt.xlabel( 'PC1' )
plt.ylabel( 'PC2' )
plt.tight_layout( )
plt.show( )

# cluster mean feature profiles
plt.figure( figsize = ( 14, 6 ) )
sns.heatmap( cluster_profile, cmap = 'YlGnBu', annot = True, fmt = '.2f' )
plt.title( 'Cluster Mean Station-Character Features' )
plt.xlabel( 'Features' )
plt.ylabel( 'Cluster' )
plt.tight_layout( )
plt.show( )



#### Feature Correlation Matrix (For KMeans Inputs)

check which station-character features are strongly correlated before clustering


In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# correlation matrix for the exact features used in clustering
feature_corr = kmeans_input[ feature_cols ].corr( )

# upper-triangle mask so we only show one half of the matrix
tri_mask = np.triu( np.ones_like( feature_corr, dtype = bool ), k = 1 )

plt.figure( figsize = ( 14, 11 ) )
sns.heatmap( 
    feature_corr,
    mask = tri_mask,
    annot = True,
    fmt = '.2f',
    cmap = 'coolwarm',
    center = 0,
    vmin = -1,
    vmax = 1,
    square = True,
    linewidths = 0.35,
    annot_kws = { 'size': 8 },
    cbar_kws = { 'label': 'Pearson r', 'shrink': 0.85 },
)
plt.title( 'KMeans Feature Correlation Matrix ( Triangle + Labels )' )
plt.tight_layout( )
plt.show( )

# strongest absolute pairwise correlations (excluding self-pairs)
corr_pairs = ( 
    feature_corr
    .where( np.triu( np.ones( feature_corr.shape ), k = 1 ).astype( bool ) )
    .stack( )
    .reset_index( )
    .rename( columns = { 'level_0': 'feature_a', 'level_1': 'feature_b', 0: 'corr' } )
)

corr_pairs[ 'abs_corr' ] = corr_pairs[ 'corr' ].abs( )
corr_pairs = corr_pairs.sort_values( 'abs_corr', ascending = False )

print( 'top correlated feature pairs:' )
corr_pairs.head( 15 )


#### 1.2b - Reduced-Feature KMeans (Second Pass)

drop features that are strongly correlated with others, then rerun KMeans to see if clustering quality improves


In [ ]:
# second KMeans pass using a simpler feature set
# logic:
# - drop mean_annual_saturation because it tracks mean_annual_oxygen closely
# - drop seasonal_amp_oxygen because it tracks seasonal_amp_water_temp closely
# - keep the rest so we still preserve chemistry + seasonality + depth structure

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

removed_feature_cols = [ 
    'mean_annual_saturation',
    'seasonal_amp_oxygen',
]

reduced_feature_cols = [ col for col in feature_cols if col not in removed_feature_cols ]

print( 'removed features (high-correlation redundancy):', removed_feature_cols )
print( 'reduced feature count:', len( reduced_feature_cols ) )

kmeans_input_reduced = station_baseline[ [ 'region', 'station' ] + reduced_feature_cols ].copy( )

# same simple fill strategy as first pass: region median, then global median
for col in reduced_feature_cols:
    kmeans_input_reduced[ col ] = kmeans_input_reduced.groupby( 'region' )[ col ].transform( lambda s: s.fillna( s.median( ) ) )
    kmeans_input_reduced[ col ] = kmeans_input_reduced[ col ].fillna( kmeans_input_reduced[ col ].median( ) )

scaler_reduced = StandardScaler( )
X_scaled_reduced = scaler_reduced.fit_transform( kmeans_input_reduced[ reduced_feature_cols ] )

# run the same K scan so we can compare shape of elbow/silhouette
k_scan_rows_reduced = [ ]

for k in range( 2, 11 ):
    km_scan = KMeans( n_clusters = k, random_state = 42, n_init = 20 )
    labels_scan = km_scan.fit_predict( X_scaled_reduced )

    k_scan_rows_reduced.append( { 
        'k': k,
        'inertia': float( km_scan.inertia_ ),
        'silhouette': float( silhouette_score( X_scaled_reduced, labels_scan ) ),
    } )

k_scan_reduced = pd.DataFrame( k_scan_rows_reduced )
k_scan_reduced[ 'inertia_drop' ] = k_scan_reduced[ 'inertia' ].shift( 1 ) - k_scan_reduced[ 'inertia' ]
k_scan_reduced[ 'inertia_drop_pct' ] = k_scan_reduced[ 'inertia_drop' ] / k_scan_reduced[ 'inertia' ].shift( 1 )

print( )
print( 'reduced-feature k scan:' )
print( k_scan_reduced.round( 4 ) )

plt.figure( figsize = ( 10, 4 ) )
plt.plot( k_scan_reduced[ 'k' ], k_scan_reduced[ 'inertia' ], marker = 'o' )
plt.title( 'Reduced Features: K vs Inertia' )
plt.xlabel( 'K' )
plt.ylabel( 'Inertia' )
plt.tight_layout( )
plt.show( )

plt.figure( figsize = ( 10, 4 ) )
plt.plot( k_scan_reduced[ 'k' ], k_scan_reduced[ 'silhouette' ], marker = 'o' )
plt.title( 'Reduced Features: K vs Silhouette Score' )
plt.xlabel( 'K' )
plt.ylabel( 'Silhouette' )
plt.tight_layout( )
plt.show( )

# keep K aligned with first pass so comparisons stay straightforward
k_clusters_reduced = k_clusters if 'k_clusters' in globals( ) else 4

kmeans_model_reduced = KMeans( n_clusters = k_clusters_reduced, random_state = 42, n_init = 20 )
kmeans_input_reduced[ 'cluster_reduced' ] = kmeans_model_reduced.fit_predict( X_scaled_reduced )

# compare silhouette at the chosen K between full and reduced feature sets
silhouette_reduced = float( silhouette_score( X_scaled_reduced, kmeans_input_reduced[ 'cluster_reduced' ] ) )

if 'X_scaled' in globals( ):
    kmeans_compare_full = KMeans( n_clusters = k_clusters_reduced, random_state = 42, n_init = 20 )
    labels_full_compare = kmeans_compare_full.fit_predict( X_scaled )
    silhouette_full_compare = float( silhouette_score( X_scaled, labels_full_compare ) )

else:
    silhouette_full_compare = np.nan

print( )
print( f'chosen K ( reduced ): { k_clusters_reduced }' )
print( 'silhouette ( full features ):', round( float( silhouette_full_compare ), 4 ) )
print( 'silhouette ( reduced features ):', round( float( silhouette_reduced ), 4 ) )

print( )
print( 'reduced-feature cluster sizes:' )
print( kmeans_input_reduced[ 'cluster_reduced' ].value_counts( ).sort_index( ) )

# save reduced clusters onto station tables for later inspection
if 'cluster_reduced' in station_baseline.columns:
    station_baseline = station_baseline.drop( columns = [ 'cluster_reduced' ] )

station_baseline = station_baseline.merge( 
    kmeans_input_reduced[ [ 'region', 'station', 'cluster_reduced' ] ],
    on = [ 'region', 'station' ],
    how = 'left',
)

if 'station_baseline_display' in globals( ):
    if 'cluster_reduced' in station_baseline_display.columns:
        station_baseline_display = station_baseline_display.drop( columns = [ 'cluster_reduced' ] )

    station_baseline_display = station_baseline_display.merge( 
        kmeans_input_reduced[ [ 'region', 'station', 'cluster_reduced' ] ],
        on = [ 'region', 'station' ],
        how = 'left',
    )

# quick PCA view of reduced-feature clusters
pca_reduced = PCA( n_components = 2 )
pcs_reduced = pca_reduced.fit_transform( X_scaled_reduced )

cluster_plot_reduced = kmeans_input_reduced[ [ 'region', 'station', 'cluster_reduced' ] ].copy( )
cluster_plot_reduced[ 'pc1' ] = pcs_reduced[ :, 0 ]
cluster_plot_reduced[ 'pc2' ] = pcs_reduced[ :, 1 ]

plt.figure( figsize = ( 11, 7 ) )
sns.scatterplot( 
    data = cluster_plot_reduced,
    x = 'pc1',
    y = 'pc2',
    hue = 'cluster_reduced',
    palette = 'tab10',
    s = 70,
    alpha = 0.9,
)
plt.title( 'Reduced-Feature KMeans Clusters ( PCA View )' )
plt.xlabel( 'PC1' )
plt.ylabel( 'PC2' )
plt.tight_layout( )
plt.show( )

station_baseline[ [ 'region', 'station', 'cluster', 'cluster_reduced' ] ].head( 20 )



#### PCA Interpretation (What the Principal Components Mean)

- **PC1** and **PC2** are weighted blends of the station-character features.
- The **explained variance ratio** says how much of total feature variation each PC captures.
- **Large absolute loadings** (positive or negative) show which features most shape a PC.
- In the scatter plot, stations close together in PC space have similar overall character.


In [ ]:
# quick PCA interpretation table
pca_variance = pd.Series( 
    pca.explained_variance_ratio_,
    index = [ f'PC{i+1}' for i in range( len( pca.explained_variance_ratio_ ) ) ],
    name = 'explained_variance_ratio',
)

pca_loadings = pd.DataFrame( 
    pca.components_.T,
    index = feature_cols,
    columns = [ f'PC{i+1}' for i in range( pca.n_components_ ) ],
)

print( 'explained variance ratio:' )
print( pca_variance.round( 3 ) )

print( )
print( 'top absolute loadings for PC1:' )
print( pca_loadings[ 'PC1' ].abs( ).sort_values( ascending = False ).head( 8 ) )

print( )
print( 'top absolute loadings for PC2:' )
print( pca_loadings[ 'PC2' ].abs( ).sort_values( ascending = False ).head( 8 ) )

pca_loadings.round( 3 )

# suggested plain-language labels from loading signs
print( )
print( 'suggested component labels:' )
print( 'PC1: saline / alkaline / deeper-structure vs high seasonal variability' )
print( 'PC2: cool-oxygenated systems vs warm-lower-oxygen systems' )



## Phase 2 — Temporal Diagnostics
Goal: characterize lag structure and trends before building predictive features

- 2.1 run STL decomposition on water temperature per station
  - annual + diurnal cycles 
  - extract trend components
  - see trends analysis for example
- 2.2 compute cross-correlations between air temp and water temp across a range of lags (0–14 days)
  - identify lag-at-peak-correlation per station
- 2.3 repeat cross-correlation for wind/precip → salinity, air temp → DO
  - reading suggests DO may be longer lag times
- 2.4 summarize lag structure by regime
  - do cold estuaries respond more slowly than warm ones?
  - other oddities? 
  - stuff we'd report on in paper/poster
- 2.5 identify stations showing trend drift in the STL trend component
  - lag candidates for regime-transition analysis
  - feeds into phase 4, btw...

In [ ]:
# more code!

## Phase 3 — Feature Engineering
Goal: build the lagged, accumulated features your models will actually use

- 3.1 construct rolling-mean atmospheric features at multiple windows
  - 24hr, 72hr, 7-day, 14-day air temp averages
- 3.2 construct rate-of-change features (first diffs) for air temp and wind speed
- 3.3 maybe measure time cyclically?
  - day-of-year as (sin, cos) pair
  - hour-of-day similarly if using 1hr data
  - this just so hours 0 and 23, or days 1 and 365 don't SEEM far apart when they're right next to each other
  - some libraries probably do this already, btw...
- 3.4 construct delta/dff features relative to station's baseline period mean (these will probably do better in a model later)
  - Δ water temp
  - Δ salinity
  - Δ oxygen (dissolved, saturation?)
  - others?

### NOTE -- Phase 3 feeds both 5 and 6.

In [ ]:
# code???

## Phase 4 — Regime Transition Detection
Goal: identify natural validation cases and a forward-projection threshold

- 4.1 for each station, compute rolling 5-year means of key variables
  - slide across the full 30-year record
- 4.2 Compare rolling means to baseline regime centroids
  - flag stations whose recent window has crossed into an adjacent regime's territory
- 4.3 manually review flagged stations for plausibility
  - data gaps, instrument changes, or genuine trend?
- 4.4 designate confirmed transitioning stations as a held-out validation set
  - do not use in model training

### NOTE -- Phase 4 must complete before Phase 5

In [ ]:
# code is the worst ...

## Phase 5 — Model Development: Air → Water Temperature
Goal: predict ΔT_water given atmospheric forcing history

# also stage 1

- 5.1 establish naive baselines
  - persistence forecast and climatological mean by day-of-year
  - record RMSE and MAE
- 5.2 train ridge/lasso regression using lagged atmospheric features
  - interpret coefficients as a sanity check
- 5.3 train gradient boosted model (XGBoost or LightGBM) on same features
  - compare to linear baseline
- 5.4 use walk-forward temporal cross-validation
  - train on years 1–N, test on N+1
  - since reading suggests standard k-fold doesn't work well with this kind of job
- 5.5 evaluate on held-out transitioning stations from phase 4
  - does the model generalize across space?
- 5.6 select best model, document feature importances
  - lag windows that dominate are a reportable finding

In [ ]:
# code code code


## Phase 6 — Model Development: Water Temp → Water Properties
Goal: predict Δsalinity, ΔDO, ΔpH given ΔT_water and atmospheric context

# also stage 2

- 6.1 for each target variable, repeat baseline 
  - linear → gradient boosted sequence from phase 5
- 6.2 use predicted Δ air temp for water as input
  - keeps the pipeline honest and propagates uncertainty
- 6.3 compare response sensitivity by regime
  - does a +2°C warming suppress dissolved oxy more in warm estuaries than cold ones?
  - prolly a yes ... quantifying it is the contribution
- 6.4 document responses as simple lookup relationships
  - e.g., per-degree sensitivities per regime 
  - these are our core scientific deliverables

In [ ]:
# code is okay, I guess ...

## Phase 7 — Nutrient Models
Goal: predict changes in phosphates, nitrates, chlorophyll given water state

# also stage 3 (if time permits)

- 7.1 use the nutrient-inclusive dataset 
  - is okay that coverage is sparser and document that explicitly
- 7.2 same pipeline/process as phase 6, but input features now include water property predictions from earlier
- 7.3 report model skill honestly 
  - nutrient prediction will be noisier; even identifying dominant drivers is a valid result
  - so far, chlorophyll (despite being most obvious to people as blooms) seemed the weakest response?

In [ ]:
# code

## Phase 8 — Forward Projection
Goal: apply response functions to CMIP6 scenarios

- 8.1 obtain CMIP6 projected Δ air temp for your estuary regions under SSP2-4.5 and SSP5-8.5
  - ESGF portal or pre-downscaled products like NASA NEX-GDDP
  - also ote that there's some evidence that CMIP6 has under-estimated climate damage (by overestimating plant growth)
  - and the new estimate thinks it might be about 11% worse than predicted
  - might factor that in (with a *note?)
- 8.2 feed projected Δ air temp into stage 1 model 
  - get projected Δ water temp by decade
- 8.3 Feed projected Δ water temp into stage 2 response functions 
  - get projected Δ salinity, Δ oxygen, Δ pH
- 8.4 Identify projected regime-crossing dates per station under each scenario 
  - *"station X transitions from cool to warm regime between 2045–2060 under SSP5-8.5"* maybe?
- 8.5 propagate and report uncertainty 
  - at minimum, show scenario spread (SSP2 vs SSP5)
  - if time permits, model confidence intervals

In [ ]:
# code, though...